# jpomop walkthrough with invented data

This notebook runs jpomop from start to end on 100 invented patients.
It shows the two input files, the checks, the conversion and the eight OMOP CDM tables that come out.

Every patient, hospital, drug code and procedure code here is invented.
The ICD-10 codes are real, but the patients and their records are invented.
The notebook does not need any hospital data, code master or Athena download.

jpomop is at an early stage (version 0.0.1) and has not been run on real hospital data.
This notebook shows what the code does with files that follow the specification, not how well it works on real files.

To run it:

```sh
git clone https://github.com/kojikojikojikoji/jpomop
cd jpomop
pip install -e . pandas jupyter
jupyter lab examples/walkthrough_en.ipynb
```

A Japanese version is in `walkthrough_ja.ipynb`.

## 1. Set up

All files go into a temporary folder, and the last cell deletes it.

In [1]:
from pathlib import Path
import tempfile

import duckdb
import pandas as pd

import jpomop

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

work = Path(tempfile.mkdtemp(prefix="jpomop_"))
print("jpomop", jpomop.__version__)

jpomop 0.0.1


## 2. Make the invented data

`generate` writes two files for June 2026.
The same seed always gives the same files, byte for byte.

- `yoshiki1.txt` is Form 1 (様式1). It has the patient, the admission, the discharge and the diagnoses of each stay.
- `ef.txt` is the EF file (入院EF統合ファイル). It has one line for each billed item on each day.

Both files are tab-separated text in Shift JIS, as the DPC manual requires. jpomop reads them as CP932, the Windows form of Shift JIS.

In [2]:
from jpomop.synth import generate

m = generate(work / "synth", n_patients=100, seed=20261012)
print(m.n_patients, "patients,", m.n_stays, "stays")
for p in (m.yoshiki1, m.ef):
    print(p.name, p.stat().st_size, "bytes,", len(p.read_bytes().splitlines()), "lines")

100 patients, 113 stays
yoshiki1.txt 66443 bytes, 821 lines
ef.txt 313518 bytes, 1707 lines


## 3. What Form 1 looks like

Form 1 is a long file, with several lines for each stay.
Each line has 17 fields: eight fixed fields, then nine payload fields.
The `code` field says what the payload fields mean on that line.

Here are the lines of the patient with `data_id` 0000000002.
The rest of this notebook follows this patient.

In [3]:
from jpomop.spec import load_spec

y1_spec = load_spec("dpc_yoshiki1")
lines = m.yoshiki1.read_bytes().decode("cp932").splitlines()
raw_y1 = pd.DataFrame([ln.split("\t") for ln in lines], columns=y1_spec.field_names())
raw_y1[raw_y1["data_id"] == "0000000002"]

,facility_code,data_id,admission_date,stay_no,summary_no,code,version,seq,payload1,payload2,payload3,payload4,payload5,payload6,payload7,payload8,payload9
6,999999999,0000000002,20260609,0,0,A000010,20140401,0,19521204,1,1000001,,,,,,
7,999999999,0000000002,20260609,0,0,A000020,20140401,0,20260609,1,,,100,0,,,
8,999999999,0000000002,20260609,0,0,A000030,20140401,0,20260617,4,4,0,,,,,
9,999999999,0000000002,20260609,0,0,A006010,20140401,0,,J189,,9000001,8002,,,,肺炎の疑い
10,999999999,0000000002,20260609,0,0,A006020,20140401,0,,J189,,9000001,8002,,,,肺炎の疑い
11,999999999,0000000002,20260609,0,0,A006030,20140401,0,,J189,,9000001,8002,,,,肺炎の疑い
12,999999999,0000000002,20260609,0,0,A006040,20140401,1,,E785,,9000103,,,,,脂質異常症


How to read these lines:

| `code` | What the line holds | Payload fields used above |
|---|---|---|
| A000010 | The patient | 1: birth date, 2: sex, 3: postcode |
| A000020 | The admission | 1: admission date, 2: where the patient came from |
| A000030 | The discharge | 1: discharge date, 2: where the patient went, 3: outcome, 4: death within 24 hours |
| A006010 | The main diagnosis | 2: ICD-10, 4: disease code, 5: modifier code, 9: name |
| A006020 | The diagnosis that led to the admission | same as A006010 |
| A006030 | The diagnosis that used the most resources | same as A006010 |
| A006040 | A diagnosis the patient already had at admission | same as A006010 |

This patient has the same diagnosis, J189 (pneumonia), on three lines, and all three lines carry the modifier code 8002.
In the DPC manual, 8002 means "suspected".
Section 8 shows what jpomop does with it.

`summary_no` is 統括診療情報番号 in the manual.
The value 0 means that the form covers the whole stay.
Section 10 shows a form with another value.

jpomop keeps the layout in a YAML file.
Each field has its position, type, length and the page of the manual where it is defined.

In [4]:
pd.DataFrame(
    [(f.col, f.name, f.name_ja, f.type, f.length, f.page) for f in y1_spec.fields],
    columns=["col", "name", "name_ja", "type", "length", "manual_page"],
).astype({"length": "Int64"})

,col,name,name_ja,type,length,manual_page
0,1,facility_code,施設コード,digits,9,22
1,2,data_id,データ識別番号,digits,10,22
2,3,admission_date,入院年月日,date8,8,22
3,4,stay_no,回数管理番号,string,<NA>,22
4,5,summary_no,統括診療情報番号,string,<NA>,22
5,6,code,コード,string,7,22
6,7,version,バージョン,date8,8,22
7,8,seq,連番,integer,<NA>,22
8,9,payload1,ペイロード1,string,<NA>,22
9,10,payload2,ペイロード2,string,<NA>,22


## 4. What the EF file looks like

The EF file has 31 fields on each line.
One line is one billed item on one day: a drug, a test, a procedure or the daily room charge.
The table below shows nine of the 31 fields for the same patient.

In [5]:
ef_spec = load_spec("dpc_ef")
lines = m.ef.read_bytes().decode("cp932").splitlines()
raw_ef = pd.DataFrame([ln.split("\t") for ln in lines], columns=ef_spec.field_names())
cols = ["data_id", "admission_date", "exec_date", "data_kubun", "receipt_code",
        "item_name", "quantity", "detail_flags", "ward_code"]
raw_ef.loc[raw_ef["data_id"] == "0000000002", cols]

,data_id,admission_date,exec_date,data_kubun,receipt_code,item_name,quantity,detail_flags,ward_code
6,0000000002,20260609,20260609,90,190999901,架空入院基本料,0000000.000,000000000000,W1
7,0000000002,20260609,20260609,60,160999901,架空検査（血液）,0000000.000,000000000000,W1
8,0000000002,20260609,20260610,90,190999901,架空入院基本料,0000000.000,000000000000,W1
9,0000000002,20260609,20260610,33,640999903,架空注射液Ｃ ５００ｍＬ,0000001.000,000000000000,W1
10,0000000002,20260609,20260610,70,170999901,架空画像診断,0000000.000,000000000000,W1
11,0000000002,20260609,20260611,90,190999901,架空入院基本料,0000000.000,000000000000,W1
12,0000000002,20260609,20260611,33,640999903,架空注射液Ｃ ５００ｍＬ,0000001.000,000000000000,W1
13,0000000002,20260609,20260611,21,620999901,架空錠Ａ ５ｍｇ,0000001.000,000000000000,W1
14,0000000002,20260609,20260612,90,190999901,架空入院基本料,0000000.000,000000000000,W1
15,0000000002,20260609,20260612,21,620999901,架空錠Ａ ５ｍｇ,0000001.000,000000000000,W1


How to read these lines:

- `exec_date` is the day of the service. The stay runs from 9 June to 17 June, so there are lines for nine days.
- `data_kubun` (データ区分) is the kind of item. 21 is an oral drug, 33 is an injection, 60 is a test, 70 is imaging and 90 is the basic inpatient charge (入院基本料).
- `receipt_code` is the code of the item. In real files it comes from the national code masters. Here the codes are invented. In the national masters, drug codes start with 6 and medical procedure codes start with 1. The invented codes follow this.
- `ward_code` is the ward on that day. This patient moves from ward W1 to ward W2 on 14 June. Each hospital chooses its own ward codes.
- `detail_flags` (行為明細区分情報) has 12 digits. The third digit marks drugs that the patient brought from home. Here all digits are 0.

The next table counts the lines in the whole file by kind of item.

In [6]:
from jpomop.spec import load_codes

kubun = load_codes("ef_data_kubun")
t = raw_ef.groupby(["data_kubun", "receipt_code", "item_name"]).size().rename("ef_lines").reset_index()
t.insert(1, "data_kubun_name", t["data_kubun"].map(kubun))
t

,data_kubun,data_kubun_name,receipt_code,item_name,ef_lines
0,21,内服,620999901,架空錠Ａ ５ｍｇ,243
1,21,内服,620999902,架空カプセルＢ,208
2,33,その他（注射）,640999903,架空注射液Ｃ ５００ｍＬ,241
3,50,手術,150999901,架空手術,34
4,60,検査・病理,160999901,架空検査（血液）,141
5,70,画像診断,170999901,架空画像診断,150
6,90,入院基本料,190999901,架空入院基本料,690


## 5. Read and check the files

`read_yoshiki1` and `read_ef` read the files and check the fields that jpomop reads against its specification.
`pivot_yoshiki1` then turns the long Form 1 into one row for each form and one row for each diagnosis.

The invented files pass those checks, so there are no reported violations.

In [7]:
from jpomop.parse.ef import read_ef
from jpomop.parse.yoshiki1 import pivot_yoshiki1, read_yoshiki1

y1 = read_yoshiki1(m.yoshiki1)
tables = pivot_yoshiki1(y1.table)
ef = read_ef(m.ef)

print("Form 1 lines:", y1.table.num_rows)
print("forms:", tables.forms.num_rows, " diagnoses:", tables.diagnoses.num_rows)
print("EF lines:", ef.table.num_rows)
print("violations:", len(y1.violations) + len(tables.violations) + len(ef.violations))

Form 1 lines: 821
forms: 113  diagnoses: 448
EF lines: 1707
violations: 0


This is the one form of our patient after the pivot.
The postcode is not kept.

In [8]:
forms = tables.forms.to_pandas()
forms[forms["data_id"] == "0000000002"].T

,1
facility_code,999999999
data_id,0000000002
admission_date,2026-06-09
stay_no,0
summary_no,0
birth_date,1952-12-04
sex,1
admission_route,1
admission_type,100
ambulance,0


The codes for sex, admission route, discharge destination and outcome come from the manual.
jpomop keeps them in small tables.
Our patient is male (1), came from home (1), moved to another hospital (4) and the outcome was "unchanged" (4).

In [9]:
for name in ["y1_sex", "y1_admission_route", "y1_discharge_to", "y1_discharge_outcome"]:
    print(name, load_codes(name))

y1_sex {'1': '男', '2': '女'}
y1_admission_route {'0': '院内の他病棟からの転棟', '1': '家庭からの入院', '4': '他の病院・診療所の病棟からの転院', '5': '介護施設・福祉施設に入所中', '8': '院内で出生', '9': 'その他'}
y1_discharge_to {'0': '院内の他病棟への転棟', '1': '家庭への退院（当院に通院）', '2': '家庭への退院（他の病院・診療所に通院）', '3': '家庭への退院（その他）', '4': '他の病院・診療所への転院', '5': '介護老人保健施設に入所', '6': '介護老人福祉施設に入所', '7': '社会福祉施設、有料老人ホーム等に入所', '8': '終了（死亡等）', '9': 'その他', 'a': '介護医療院', 'b': '未転棟、未退院'}
y1_discharge_outcome {'1': '治癒・軽快', '3': '寛解', '4': '不変', '5': '増悪', '6': '最も医療資源を投入した傷病による死亡', '7': '最も医療資源を投入した傷病以外による死亡', '9': 'その他'}


These are the diagnoses of the same form.
`role` says which line of Form 1 the diagnosis came from.
`suspected` is true when the modifier code 8002 is present.

In [10]:
dx = tables.diagnoses.to_pandas()
dx.loc[dx["data_id"] == "0000000002",
       ["role", "seq", "icd10", "disease_code", "modifiers", "suspected", "disease_name"]]

,role,seq,icd10,disease_code,modifiers,suspected,disease_name
3,dx_main,0,J189,9000001,8002,True,肺炎の疑い
4,dx_trigger,0,J189,9000001,8002,True,肺炎の疑い
5,dx_resource1,0,J189,9000001,8002,True,肺炎の疑い
6,dx_comorbid,1,E785,9000103,NaN,False,脂質異常症


## 6. What a broken file looks like

To see the checks at work, the next cell makes copies of the files with four mistakes:

1. a birth date of 30 February
2. a sex code of 9, which is not in the manual
3. a Form 1 line with 16 fields
4. an EF line with a service date after the discharge date

In [11]:
bad_lines = m.yoshiki1.read_bytes().decode("cp932").splitlines()

a = bad_lines[0].split("\t")   # first line: birth date and sex
a[8] = "19520230"               # 30 February
a[9] = "9"                      # sex code that is not in the manual
bad_lines[0] = "\t".join(a)

b = bad_lines[1].split("\t")   # second line: drop the last field
bad_lines[1] = "\t".join(b[:-1])

bad_y1 = work / "yoshiki1_bad.txt"
bad_y1.write_bytes(("\r\n".join(bad_lines) + "\r\n").encode("cp932"))

bad_ef_lines = m.ef.read_bytes().decode("cp932").splitlines()
c = bad_ef_lines[0].split("\t")
c[23] = "20260701"              # service date after discharge
bad_ef_lines[0] = "\t".join(c)
bad_ef = work / "ef_bad.txt"
_ = bad_ef.write_bytes(("\r\n".join(bad_ef_lines) + "\r\n").encode("cp932"))

The checks report each mistake with the file, the line, the field, the value and the rule that was broken.

In [12]:
from jpomop.parse.cross import check_cross_file

r1 = read_yoshiki1(bad_y1)
t1 = pivot_yoshiki1(r1.table)
r2 = read_ef(bad_ef)
found = r1.violations + t1.violations + r2.violations
found += check_cross_file(r2.table, t1.forms, ef_file=bad_ef.name, y1_file=bad_y1.name)

pd.DataFrame([vars(v) for v in found])

,file,line,field,value,rule,message
0,yoshiki1_bad.txt,2,*,16,n_fields,expected 17 fields
1,yoshiki1_bad.txt,1,A000010.1:birth_date,19520230,date_invalid,not a calendar date
2,yoshiki1_bad.txt,1,A000010.2:sex,9,code_not_in_table,not in the code table
3,ef_bad.txt,1,exec_date,2026-07-01,exec_outside_stay,date of service is outside the stay


The same checks run from the command line:

```sh
jpomop validate --yoshiki1 yoshiki1.txt --ef ef.txt --out violations.csv
```

The command ends with status 1 when it finds a violation.
jpomop checks only the fields that it reads.
It is not a replacement for the official checking tool.

## 7. Convert to the OMOP CDM

`run_etl` reads both files and writes the CDM tables into one DuckDB file.
The command line does the same with `jpomop convert`.

The report shows the number of rows in each table.
It also shows what was left out and why.

In [13]:
from jpomop.etl import run_etl

cdm_path = work / "cdm.duckdb"
report = run_etl([m.yoshiki1], [m.ef], cdm_path)

print(pd.Series(report.rows, name="rows").to_string())
print()
print("forms not used:", report.skipped_forms)
print("rows left out:", report.skipped_rows)
print("violations:", report.n_violations, " consistency problems:", report.invariant_problems)

person                  100
observation_period      100
visit_occurrence        113
visit_detail            138
death                     5
condition_occurrence    213
drug_exposure           692
procedure_occurrence    325

forms not used: {}
rows left out: {'condition_occurrence': {'duplicate_in_stay': 208, 'suspected': 27}, 'drug_exposure': {}}
violations: 0  consistency problems: []


How to read the report:

- 100 patients had 113 stays. So there are 100 rows in `person` and 113 rows in `visit_occurrence`.
- `duplicate_in_stay` is 208. Form 1 often gives the same diagnosis on three lines, as we saw. jpomop writes one row for each ICD-10 code in a stay, not three.
- `suspected` is 27. These are the diagnosis lines with the modifier code 8002. They are not written to `condition_occurrence`.
- "consistency problems" lists rows that break basic rules, for example an event outside the observation period. The list is empty.

The next sections look at each table for our patient.

## 8. Follow one patient through the tables

### person

One row for each patient.
`person_source_value` is a hash of the hospital code and the patient number, so the original number is not in the output.
`gender_concept_id` 8507 is the OMOP concept for male.
These DPC input files have no race or ethnicity fields, so those concept IDs are 0.

In [14]:
con = duckdb.connect(str(cdm_path), read_only=True)

con.sql("""
    SELECT person_id, person_source_value, gender_source_value, gender_concept_id,
           year_of_birth, month_of_birth, day_of_birth
    FROM person WHERE person_id = 2
""").df()

,person_id,person_source_value,gender_source_value,gender_concept_id,year_of_birth,month_of_birth,day_of_birth
0,2,dcc269630e6028ae,1,8507,1952,12,4


### visit_occurrence

One row for each stay.
`visit_concept_id` 9201 is "Inpatient Visit".
`visit_type_concept_id` 32810 is "Claim".
The source values 1 and 4 are the DPC codes for "came from home" and "moved to another hospital".

In [15]:
con.sql("""
    SELECT visit_occurrence_id, visit_concept_id, visit_start_date, visit_end_date,
           visit_type_concept_id, admitted_from_source_value, discharged_to_source_value
    FROM visit_occurrence WHERE person_id = 2
""").df()

,visit_occurrence_id,visit_concept_id,visit_start_date,visit_end_date,visit_type_concept_id,admitted_from_source_value,discharged_to_source_value
0,2,9201,2026-06-09,2026-06-17,32810,1,4


### visit_detail

One row for each run of days on the same ward.
Our patient was on ward W1 for five days and then on ward W2 for four days.
The second row points back to the first row through `preceding_visit_detail_id`.

In [16]:
con.sql("""
    SELECT visit_detail_id, visit_occurrence_id, visit_detail_start_date,
           visit_detail_end_date, visit_detail_source_value, preceding_visit_detail_id
    FROM visit_detail WHERE person_id = 2 ORDER BY visit_detail_start_date
""").df()

,visit_detail_id,visit_occurrence_id,visit_detail_start_date,visit_detail_end_date,visit_detail_source_value,preceding_visit_detail_id
0,2,2,2026-06-09,2026-06-13,W1,<NA>
1,3,2,2026-06-14,2026-06-17,W2,2


jpomop gets the wards from the EF file, not from Form 1.
This is the ward code on each day of the stay in the EF file.
The two rows above match these ward codes.

In [17]:
(raw_ef[raw_ef["data_id"] == "0000000002"]
 .groupby("exec_date")["ward_code"].agg(lambda s: ",".join(sorted(set(s))))
 .to_frame().T)

exec_date,20260609,20260610,20260611,20260612,20260613,20260614,20260615,20260616,20260617
ward_code,W1,W1,W1,W1,W1,W2,W2,W2,W2


### condition_occurrence

Form 1 gave four diagnosis lines for this patient.
Three of them were suspected pneumonia.
Only the fourth, E785, is in the table.

In [18]:
con.sql("""
    SELECT condition_occurrence_id, condition_start_date, condition_source_value,
           condition_concept_id, condition_status_source_value, condition_status_concept_id
    FROM condition_occurrence WHERE person_id = 2
""").df()

,condition_occurrence_id,condition_start_date,condition_source_value,condition_concept_id,condition_status_source_value,condition_status_concept_id
0,2,2026-06-09,E785,0,dx_comorbid,32907


Form 1 has no date for each diagnosis.
So `condition_start_date` is the admission date.
It is not the date when the disease began.

`condition_status_concept_id` says which line of Form 1 the diagnosis came from.
32907 is "Secondary admission diagnosis", which jpomop uses for a diagnosis the patient already had at admission.
This choice of concept is my own reading. It is not an official mapping.

For comparison, here is patient 0000000001.
The diagnosis I639 is on three lines and is not suspected.
It becomes one row, and the row keeps the role of the main diagnosis.

In [19]:
print(dx.loc[dx["data_id"] == "0000000001", ["role", "icd10", "suspected"]].to_string(index=False))
con.sql("""
    SELECT condition_start_date, condition_source_value, condition_status_source_value
    FROM condition_occurrence WHERE person_id = 1
""").df()

        role icd10  suspected
     dx_main  I639      False
  dx_trigger  I639      False
dx_resource1  I639      False


,condition_start_date,condition_source_value,condition_status_source_value
0,2026-06-01,I639,dx_main


### drug_exposure

One row for each drug line in the EF file.
The start date is the service date in the EF file.
`quantity` comes from the EF file.
`route_source_value` keeps the kind of item (21 oral, 33 injection), and `dose_unit_source_value` keeps the unit code.

`drug_concept_id` is 0 because no vocabulary was given.
Section 11 shows the conversion with a vocabulary.

In [20]:
con.sql("""
    SELECT drug_exposure_id, drug_exposure_start_date, drug_source_value, drug_concept_id,
           quantity, route_source_value, dose_unit_source_value, visit_occurrence_id
    FROM drug_exposure WHERE person_id = 2 ORDER BY drug_exposure_start_date, drug_exposure_id
""").df()

,drug_exposure_id,drug_exposure_start_date,drug_source_value,drug_concept_id,quantity,route_source_value,dose_unit_source_value,visit_occurrence_id
0,4,2026-06-10,640999903,0,1.0,33,033,2
1,5,2026-06-11,640999903,0,1.0,33,033,2
2,6,2026-06-11,620999901,0,1.0,21,009,2
3,7,2026-06-12,620999901,0,1.0,21,009,2
4,8,2026-06-12,620999902,0,2.0,21,009,2
5,9,2026-06-14,620999902,0,2.0,21,009,2
6,10,2026-06-14,620999901,0,1.0,21,009,2
7,11,2026-06-16,620999901,0,1.0,21,009,2
8,12,2026-06-16,620999902,0,2.0,21,009,2


### procedure_occurrence

One row for each EF line whose code starts with 1, except room charges and meals.
For this patient these are two lines for blood tests and three lines for imaging.
The nine room-charge lines are not in the table.

`procedure_concept_id` is always 0 in this version.
jpomop does not map Japanese procedure codes to standard concepts yet.

In [21]:
con.sql("""
    SELECT procedure_occurrence_id, procedure_date, procedure_source_value,
           procedure_concept_id, visit_occurrence_id
    FROM procedure_occurrence WHERE person_id = 2 ORDER BY procedure_date
""").df()

,procedure_occurrence_id,procedure_date,procedure_source_value,procedure_concept_id,visit_occurrence_id
0,2,2026-06-09,160999901,0,2
1,3,2026-06-10,170999901,0,2
2,4,2026-06-14,160999901,0,2
3,5,2026-06-15,170999901,0,2
4,6,2026-06-16,170999901,0,2


### observation_period

One row for each patient, from the first admission to the last discharge in the files.
Patient 5 has two stays, so the period covers both.

These inpatient DPC files do not show whether a patient is observed between two stays.
Keep this in mind for any study that counts time at risk.

In [22]:
con.sql("""
    SELECT o.person_id, o.observation_period_start_date, o.observation_period_end_date,
           v.visit_occurrence_id, v.visit_start_date, v.visit_end_date
    FROM observation_period o JOIN visit_occurrence v USING (person_id)
    WHERE o.person_id = 5 ORDER BY v.visit_start_date
""").df()

,person_id,observation_period_start_date,observation_period_end_date,visit_occurrence_id,visit_start_date,visit_end_date
0,5,2026-06-06,2026-06-21,5,2026-06-06,2026-06-12
1,5,2026-06-06,2026-06-21,6,2026-06-18,2026-06-21


## 9. Deaths

The discharge outcome in these Form 1 files records deaths in hospital, not deaths after discharge.
jpomop writes a `death` row when the discharge outcome in Form 1 is 6 or 7.
The death date is the discharge date.

In [23]:
con.sql("""
    SELECT d.person_id, d.death_date, d.death_type_concept_id,
           v.visit_start_date, v.visit_end_date, v.discharged_to_source_value
    FROM death d JOIN visit_occurrence v
      ON v.person_id = d.person_id AND v.visit_end_date = d.death_date
    ORDER BY d.person_id
""").df()

,person_id,death_date,death_type_concept_id,visit_start_date,visit_end_date,discharged_to_source_value
0,4,2026-06-05,32810,2026-06-02,2026-06-05,8
1,13,2026-06-09,32810,2026-06-05,2026-06-09,8
2,21,2026-06-10,32810,2026-06-06,2026-06-10,8
3,25,2026-06-21,32810,2026-06-20,2026-06-21,8
4,98,2026-06-08,32810,2026-06-02,2026-06-08,8


These are the Form 1 rows behind the five deaths.
Outcome 6 means death from the diagnosis that used the most resources.
Outcome 7 means death from another cause.
Deaths after discharge are not recorded in these Form 1 files, so they are not in the `death` table.

In [24]:
forms.loc[forms["discharge_outcome"].isin(["6", "7"]),
          ["data_id", "admission_date", "discharge_date", "discharge_outcome"]]

,data_id,admission_date,discharge_date,discharge_outcome
3,0000000004,2026-06-02,2026-06-05,7
15,0000000013,2026-06-05,2026-06-09,7
23,0000000021,2026-06-06,2026-06-10,6
28,0000000025,2026-06-20,2026-06-21,7
110,0000000098,2026-06-02,2026-06-08,7


## 10. Forms that are counted but not converted

A hospital can write more than one Form 1 for one stay.
For example, when a patient moves between ward groups, the hospital writes one form for the whole stay and one form for each ward group.
The whole-stay form has `summary_no` 0.
The ward-group forms have 1, 2 and so on.

The invented data has only whole-stay forms.
So the next cell copies the form of patient 0000000001, changes `summary_no` to 1 and adds it to the file.
This is a rough stand-in for a ward-group form.

In [25]:
src = m.yoshiki1.read_bytes().decode("cp932").splitlines()
extra = []
for ln in src:
    f = ln.split("\t")
    if f[1] == "0000000001":
        f[4] = "1"              # 統括診療情報番号: 0 -> 1 (ward-group form)
        extra.append("\t".join(f))

y1_more = work / "yoshiki1_with_ward_group_form.txt"
y1_more.write_bytes(("\r\n".join(src + extra) + "\r\n").encode("cp932"))

report2 = run_etl([y1_more], [m.ef], work / "cdm2.duckdb")
print("forms not used:", report2.skipped_forms)
print("visit_occurrence rows:", report2.rows["visit_occurrence"],
      " condition_occurrence rows:", report2.rows["condition_occurrence"])

forms not used: {'summary_no=1': 1}
visit_occurrence rows: 113  condition_occurrence rows: 213


The extra form is counted under "forms not used".
The numbers of stays and diagnoses are the same as before (113 and 213), so the stay was not counted twice.

jpomop treats the forms for joined readmissions (A and B) in the same way.
It uses the day-90 interim form (Z) only when a stay has no whole-stay form yet.
I have not checked these rules on real files.

## 11. Concept IDs and the vocabulary

So far every concept ID for a diagnosis, drug or procedure is 0.
The original codes are in the source value columns.

In [26]:
con.sql("""
    SELECT 'condition_occurrence' AS cdm_table, count(*) AS n_rows,
           count(*) FILTER (WHERE condition_concept_id = 0) AS concept_id_0
    FROM condition_occurrence
    UNION ALL SELECT 'drug_exposure', count(*),
           count(*) FILTER (WHERE drug_concept_id = 0) FROM drug_exposure
    UNION ALL SELECT 'procedure_occurrence', count(*),
           count(*) FILTER (WHERE procedure_concept_id = 0) FROM procedure_occurrence
""").df()

,cdm_table,n_rows,concept_id_0
0,condition_occurrence,213,213
1,drug_exposure,692,692
2,procedure_occurrence,325,325


To fill in the concept IDs, jpomop needs three files that you download yourself:

- the Athena vocabularies (ICD10, SNOMED, RxNorm, RxNorm Extension)
- the drug master (医薬品マスター), which links each drug's receipt code to its NHI drug price code
- the table by Kimura et al. (2025), which links NHI drug price codes to OMOP concepts

This notebook cannot ship those files.
So the next cell writes three tiny files of the same shape.
All concept IDs in them are invented and start with 9000.
They are not taken from Athena.

The invented files cover only 8 of the 11 ICD-10 codes and 2 of the 3 drugs.
This also shows what happens to a code with no match.

In [27]:
athena = work / "athena_invented"
athena.mkdir()

# (ICD-10 code, invented ICD10 concept ID, invented standard concept ID)
icd = [("J18.9", 900000001, 900000101), ("I63.9", 900000002, 900000102),
       ("I50.0", 900000003, 900000103), ("K80.2", 900000004, 900000104),
       ("S72.0", 900000005, 900000105), ("C18.9", 900000006, 900000106),
       ("E11.9", 900000007, 900000107), ("I10", 900000008, 900000108)]

concept = ["concept_id\tconcept_name\tdomain_id\tvocabulary_id\tconcept_class_id\t"
           "standard_concept\tconcept_code\tvalid_start_date\tvalid_end_date\tinvalid_reason"]
relationship = ["concept_id_1\tconcept_id_2\trelationship_id\tvalid_start_date\t"
                "valid_end_date\tinvalid_reason"]
for code, source_id, standard_id in icd:
    concept.append(f"{source_id}\tinvented {code}\tCondition\tICD10\tICD10 code\t\t{code}"
                   "\t19700101\t20991231\t")
    concept.append(f"{standard_id}\tinvented standard for {code}\tCondition\tSNOMED\t"
                   f"Clinical Finding\tS\tX{standard_id}\t19700101\t20991231\t")
    relationship.append(f"{source_id}\t{standard_id}\tMaps to\t19700101\t20991231\t")
(athena / "CONCEPT.csv").write_text("\n".join(concept) + "\n", encoding="utf-8")
(athena / "CONCEPT_RELATIONSHIP.csv").write_text("\n".join(relationship) + "\n", encoding="utf-8")

# A file shaped like the drug master: 42 fields, receipt code in field 3,
# NHI drug price code in field 32. Both codes are invented.
pairs = {"620999901": "9999001F1011", "620999902": "9999002M1011", "640999903": "9999400A1011"}
rows = []
for receipt, price_code in pairs.items():
    fields = ["0", "Y", receipt] + [""] * 39
    fields[31] = price_code
    rows.append(",".join(f'"{x}"' for x in fields))
drug_master = work / "y_invented.csv"
drug_master.write_bytes(("\r\n".join(rows) + "\r\n").encode("cp932"))

# A file shaped like rxnorm_mapping.tsv. The third drug has no match on purpose.
mapping = work / "rxnorm_mapping_invented.tsv"
_ = mapping.write_text("Category\tNHIDPLC \tOSV_ID\tOSV_NAME\n"
                   "PO\t9999001F1011\t900001001\tinvented drug A\n"
                   "PO\t9999002M1011\t900001002\tinvented drug B\n"
                   "Injection\t9999400A1011\tN/A\tN/A\n", encoding="utf-8")

`build_vocab` joins the three files into one small DuckDB file.

DPC writes ICD-10 codes without a dot and sometimes with a fifth character.
jpomop tries the most specific code first and then shorter ones.
S7200 is not in the invented vocabulary as S72.00, so it falls back to S72.0.
N390 has no match, so its concept ID is 0.

In [28]:
from jpomop.vocab import Vocab, build_vocab, icd10_candidates

vocab_path = work / "vocab.duckdb"
vr = build_vocab(athena, [drug_master], mapping, vocab_path)
print(vr)

v = Vocab.load(vocab_path)
for code in ["J189", "S7200", "N390"]:
    print(code, icd10_candidates(code), "->", v.map_icd10(code))
for code in ["620999901", "640999903"]:
    print(code, "->", v.map_drug(code))

VocabBuildReport(icd10_codes=8, icd10_mapped=8, receipt_drugs=3, receipt_drugs_mapped=2)
J189 ['J18.9', 'J18'] -> ConceptMatch(concept_id=900000101, source_concept_id=900000001, source_value='J189')
S7200 ['S72.00', 'S72.0', 'S72'] -> ConceptMatch(concept_id=900000105, source_concept_id=900000005, source_value='S7200')
N390 ['N39.0', 'N39'] -> ConceptMatch(concept_id=0, source_concept_id=0, source_value='N390')
620999901 -> ConceptMatch(concept_id=900001001, source_concept_id=0, source_value='620999901')
640999903 -> ConceptMatch(concept_id=0, source_concept_id=0, source_value='640999903')


Now the same conversion with the vocabulary.
`condition_source_concept_id` is the ICD-10 concept.
`condition_concept_id` is the standard concept that it maps to.

In [29]:
cdm3 = work / "cdm_with_vocab.duckdb"
report3 = run_etl([m.yoshiki1], [m.ef], cdm3, vocab=vocab_path)

con3 = duckdb.connect(str(cdm3), read_only=True)
con3.sql("""
    SELECT condition_source_value, condition_source_concept_id, condition_concept_id,
           count(*) AS n_rows
    FROM condition_occurrence GROUP BY ALL ORDER BY condition_source_value
""").df()

,condition_source_value,condition_source_concept_id,condition_concept_id,n_rows
0,C189,900000006,900000106,18
1,E119,900000007,900000107,25
2,E785,0,0,28
3,I10,900000008,900000108,28
4,I500,900000003,900000103,17
5,I639,900000002,900000102,9
6,J189,900000001,900000101,20
7,K802,900000004,900000104,15
8,N189,0,0,28
9,N390,0,0,14


`coverage` counts how many rows and how many different codes got a nonzero concept ID.
It also lists the codes that did not, with the most frequent first.
The command line shows the same with `jpomop report`.

With real vocabularies this table would show which codes most need mapping.
The percentages below only reflect the invented files of this notebook.
They say nothing about real data.

In [30]:
from jpomop.report import coverage

pd.DataFrame(
    [(c.table, c.rows, c.mapped_rows, f"{c.row_rate:.0%}", c.codes, c.mapped_codes, c.top_unmapped)
     for c in coverage(cdm3)],
    columns=["cdm_table", "rows", "mapped_rows", "row_rate", "codes", "mapped_codes", "unmapped_codes"],
)

,cdm_table,rows,mapped_rows,row_rate,codes,mapped_codes,unmapped_codes
0,condition_occurrence,213,143,67%,11,8,"[(E785, 28), (N189, 28), (N390, 14)]"
1,drug_exposure,692,451,65%,3,2,"[(640999903, 241)]"
2,procedure_occurrence,325,0,0%,3,0,"[(170999901, 150), (160999901, 141), (15099990..."


## 12. Four questions asked of the tables

The tables are in a DuckDB file, so plain SQL works.
Because the data is invented, the answers are only examples of how the tables join.

### How long are the stays for each main diagnosis?

`condition_status_concept_id` 32902 marks the main diagnosis.

In [31]:
con.sql("""
    SELECT c.condition_source_value AS main_diagnosis_icd10,
           count(*) AS stays,
           round(avg(v.visit_end_date - v.visit_start_date + 1), 1) AS mean_days,
           min(v.visit_end_date - v.visit_start_date + 1) AS min_days,
           max(v.visit_end_date - v.visit_start_date + 1) AS max_days
    FROM visit_occurrence v
    JOIN condition_occurrence c USING (visit_occurrence_id)
    WHERE c.condition_status_concept_id = 32902
    GROUP BY 1 ORDER BY stays DESC
""").df()

,main_diagnosis_icd10,stays,mean_days,min_days,max_days
0,J189,20,6.5,3,10
1,C189,18,6.4,2,10
2,I500,17,5.9,2,10
3,K802,15,6.1,2,10
4,N390,14,6.1,2,10
5,S7200,11,5.5,2,10
6,I639,9,4.7,2,8


The stays add up to 104, not 113.
The other nine stays have a suspected main diagnosis, so they have no main diagnosis row.
This is a direct result of leaving suspected diagnoses out.
A study that needs every stay should start from `visit_occurrence`, not from the diagnosis.

### How many patients got each drug, and on how many days?

In [32]:
con.sql("""
    SELECT d.drug_source_value,
           count(DISTINCT d.person_id) AS persons,
           count(DISTINCT d.visit_occurrence_id) AS stays,
           count(*) AS rows_,
           count(DISTINCT (d.person_id, d.drug_exposure_start_date)) AS person_days
    FROM drug_exposure d GROUP BY 1 ORDER BY 1
""").df()

,drug_source_value,persons,stays,rows_,person_days
0,620999901,92,104,243,243
1,620999902,90,97,208,208
2,640999903,93,103,241,241


### How many stays include a ward change?

In [33]:
con.sql("""
    SELECT wards_in_stay, count(*) AS stays
    FROM (SELECT visit_occurrence_id, count(*) AS wards_in_stay
          FROM visit_detail GROUP BY 1)
    GROUP BY 1 ORDER BY 1
""").df()

,wards_in_stay,stays
0,1,88
1,2,25


### Which patients were admitted more than once, and how many days after the last discharge?

In [34]:
con.sql("""
    SELECT person_id, visit_occurrence_id, visit_start_date, visit_end_date,
           visit_start_date - lag(visit_end_date) OVER (PARTITION BY person_id
                                                         ORDER BY visit_start_date) AS days_since_last_discharge
    FROM visit_occurrence
    QUALIFY count(*) OVER (PARTITION BY person_id) > 1
    ORDER BY person_id, visit_start_date
    LIMIT 8
""").df()

,person_id,visit_occurrence_id,visit_start_date,visit_end_date,days_since_last_discharge
0,5,5,2026-06-06,2026-06-12,<NA>
1,5,6,2026-06-18,2026-06-21,6
2,11,12,2026-06-01,2026-06-02,<NA>
3,11,13,2026-06-10,2026-06-19,8
4,12,14,2026-06-10,2026-06-17,<NA>
5,12,15,2026-06-20,2026-06-26,3
6,24,27,2026-06-03,2026-06-05,<NA>
7,24,28,2026-06-19,2026-06-24,14


## 13. Parquet output

The same tables can be written as Parquet files, one file for each table.

In [35]:
report4 = run_etl([m.yoshiki1], [m.ef], work / "cdm_parquet", fmt="parquet")
sorted(p.name for p in (work / "cdm_parquet").iterdir())

['condition_occurrence.parquet',
 'death.parquet',
 'drug_exposure.parquet',
 'observation_period.parquet',
 'person.parquet',
 'procedure_occurrence.parquet',
 'visit_detail.parquet',
 'visit_occurrence.parquet']

## 14. What this notebook does not show

- It does not show how jpomop works on real hospital files. I have not run it on real data.
- It does not show real mapping rates. The vocabulary in section 11 is invented.
- Procedures have no standard concepts yet.
- Costs and points are not converted.
- Outpatient files, the D file and the H file are not read.
- The handling of ward-group forms, readmission forms and interim forms follows my reading of the manual. It needs checking on real files.

If you work with DPC data and can run jpomop where your data is, counts and error messages would help.
You do not need to share any data.
Issues are open at https://github.com/kojikojikojikoji/jpomop/issues

The cell below closes the files and deletes the temporary folder.

In [36]:
con.close()
con3.close()
import shutil

shutil.rmtree(work)